In [4]:
from evaluation_functions import plot_pv_field
import json
import torch
from pathlib import Path
import sys
from torch_geometric.loader import DataLoader

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from shared.meshgraphnet_functions import MeshDataset, MeshGraphNet

In [7]:
sample_name = "cone_010"
model_name = "example_model"
dataset_name = "example_dataset"

with open(rf"C:\PBF-MeshGraphNet\models\{model_name}\model_config.json", "r") as f:
    config = json.load(f)

latent_size = config["latent_size"]
num_processor_blocks = config["num_processor_blocks"]

model_path = rf"C:\PBF-MeshGraphNet\models\{model_name}\{model_name}.pt"
train_path = rf"C:\PBF-MeshGraphNet\datasets\{dataset_name}\train"
test_path = rf"C:\PBF-MeshGraphNet\datasets\{dataset_name}\test"
normalization_path = rf"C:\PBF-MeshGraphNet\models\{model_name}\{model_name}_normalization.json"
external_path = rf"C:\PBF-MeshGraphNet\datasets\external_samples\external_samples"


with open(normalization_path, 'r') as f:
    normalization_data = json.load(f)
    feature_mean = normalization_data['feature_mean']
    feature_std = normalization_data['feature_std']
    continuous_feature_columns = normalization_data['continuous_feature_columns']

test_dataset = MeshDataset(test_path, feature_mean, feature_std, continuous_feature_columns)
train_dataset = MeshDataset(train_path, feature_mean, feature_std, continuous_feature_columns)
external_dataset = MeshDataset(external_path, feature_mean, feature_std, continuous_feature_columns)
input_size = test_dataset.num_features

checkpoint = torch.load(model_path, map_location = "cpu")
model = MeshGraphNet(latent_size = latent_size, node_input_size = input_size, num_processor_blocks = num_processor_blocks)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

MeshGraphNet(
  (node_encoder): MLP(
    (linear1): Linear(in_features=30, out_features=64, bias=True)
    (linear2): Linear(in_features=64, out_features=64, bias=True)
    (linear3): Linear(in_features=64, out_features=64, bias=True)
    (relu): ReLU()
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (edge_encoder): MLP(
    (linear1): Linear(in_features=4, out_features=64, bias=True)
    (linear2): Linear(in_features=64, out_features=64, bias=True)
    (linear3): Linear(in_features=64, out_features=64, bias=True)
    (relu): ReLU()
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (processor_blocks): ModuleList(
    (0-9): 10 x ProcessorBlock(
      (edge_mlp): MLP(
        (linear1): Linear(in_features=192, out_features=64, bias=True)
        (linear2): Linear(in_features=64, out_features=64, bias=True)
        (linear3): Linear(in_features=64, out_features=64, bias=True)
        (relu): ReLU()
        (norm): LayerNor

In [8]:
sample_dict = {}

external_dataset = MeshDataset(external_path, feature_mean, feature_std, continuous_feature_columns)

for dataset in [train_dataset, test_dataset, external_dataset]:
    for i in range(len(dataset)):
        sample_dict[dataset[i]["sample_name"]] = dataset[i]

In [9]:
sample_name = 'cone_010'

sample = sample_dict[sample_name]

print(f"Sample: {sample_name}")

plot_pv_field(
    model=model,
    graph=sample,
    device="cpu",
    plot = "error",
    plot_abs = True
)

Sample: cone_010


Widget(value='<iframe src="http://localhost:56255/index.html?ui=P_0x2ece09ad250_0&reconnect=auto" class="pyvis…